## Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Apache Iceberg Table with OCI Native Protocol

This notebook demonstrates how to create and interact with an **Apache Iceberg** table stored in **OCI Object Storage**, using the native `oci://` protocol via a **Hadoop Catalog**.

---

## What is Apache Iceberg?

Apache Iceberg is an open **table format** for large-scale analytic datasets. It is not a file format (like Parquet or Avro) — it sits on top of file formats and adds powerful features that raw files alone cannot provide such as Schema evolution, Time travel, ACID transactions, just to name a few.

Iceberg stores its own **metadata layer** alongside the data files. This metadata tracks snapshots, schema versions, partition layout, and data file locations.

---

## What is a Hadoop Catalog?

Iceberg requires a **catalog** to track where tables are and what their current snapshot is. There are several catalog types:

- **Hadoop Catalog**: Stores metadata directly in the filesystem (Object Storage). Simple, no external service needed. This is what this notebook uses.
- **Hive Metastore**: Uses an external Hive Metastore service for catalog management.
- **REST Catalog**: Uses a REST API backed by a catalog server. Enables centralized governance, fine-grained access control, and multi-engine interoperability.
- **Master Catalog**: The built-in catalog provided by OCI AI Data Platform. Fully managed, integrated with OCI security, and the recommended option for production workloads on the platform.

This notebook uses the **Hadoop Catalog** — metadata is written directly into the OCI bucket alongside the data. This is the simplest setup and works great for getting started or for simple architectures.

---

## Why OCI Native Protocol (`oci://`)?

OCI Object Storage can be accessed via two protocols in AI Data Platform (running Spark enabled clusters):

- **S3 Compatible API** (`s3a://`): Requires S3A JARs, endpoint configuration, and credentials setup.
- **OCI Native (`oci://`)**: Works out of the box in the AI Data Platform Workbench. Performs authentication automatically based on your user or group permissions.

This notebook uses `oci://` — no extra JARs or credential setup needed.

Note that all metadata in the catalog will reference `oci://` paths, not `s3a://`.

---

## Prerequisites

- Running in OCI AI Data Platform Workbench
- An OCI Object Storage bucket already created
- A folder (prefix) created inside the bucket to serve as the Iceberg warehouse
- Permissions already set on OCI tenancy for that bucket.Example of statements: 
    - `Allow dynamic-group <your-dynamic-group> to manage objects in compartment <compartment> where target.bucket.name='<bucket-name>'`
    - `Allow dynamic-group <your-dynamic-group> to manage buckets in compartment <compartment> where target.bucket.name='<bucket-name>'`


## Step 1: Verify Spark Session

In AI Data Platform Workbench, a Spark session (`spark`) is automatically available when running on a Spark-enabled cluster. Let's confirm it's active before proceeding.

In [ ]:
print(f"Spark version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")

## Step 2: Configuration

Fill in the variables below with your OCI environment details.

> **Tip:** The warehouse path follows the format `oci://bucket@namespace/folder`. The namespace is your OCI tenancy's Object Storage namespace, visible in the OCI Console under Object Storage settings.

In [ ]:
# ── OCI Object Storage ────────────────────────────────────────────────────────
# Your tenancy's Object Storage namespace (found in OCI Console > Object Storage)
OCI_NAMESPACE = "<YOUR_NAMESPACE>"

# Region where your bucket is located
OCI_REGION = "us-ashburn-1"

# Name of the OCI bucket where Iceberg data will be stored
BUCKET_NAME = "<YOUR_BUCKET_NAME>"

# ── Iceberg Warehouse ─────────────────────────────────────────────────────────
# The warehouse is the root folder inside the bucket where Iceberg will store
# all its databases, tables, data files, and metadata.
WAREHOUSE_NAME = "iceberg-warehouse"

# Iceberg warehouse path using OCI native protocol: oci://bucket@namespace/path
WAREHOUSE_PATH = f"oci://{BUCKET_NAME}@{OCI_NAMESPACE}/{WAREHOUSE_NAME}"

# ── Catalog, Database, and Table names ────────────────────────────────────────
# The catalog is the top-level namespace registered in Spark.
# In Iceberg SQL, tables are always referenced as: catalog.database.table
CATALOG_NAME = "oci_catalog"

# A database (also called namespace) groups related tables together.
DATABASE_NAME = "<YOUR_DATABASE>"

# The table name
TABLE_NAME = "<YOUR_TABLE>"

# Fully qualified table name used in all SQL statements
FULL_TABLE_NAME = f"{CATALOG_NAME}.{DATABASE_NAME}.{TABLE_NAME}"

print(f"Warehouse : {WAREHOUSE_PATH}")
print(f"Table     : {FULL_TABLE_NAME}")

## Step 3: Configure the Iceberg Catalog in Spark

Spark does not know about Iceberg catalogs by default. We need to register our Hadoop Catalog by setting Spark configuration properties at runtime.

These three properties are the minimum required:
- `spark.sql.catalog.<name>`: the Iceberg catalog implementation class
- `spark.sql.catalog.<name>.type`: catalog type (`hadoop`, `hive`, or `rest`)
- `spark.sql.catalog.<name>.warehouse`: the root path for all table data and metadata

In [ ]:
# Register the Iceberg catalog in Spark using the Hadoop catalog type.
# Authentication to OCI Object Storage is handled automatically based on your Workbench permissions.
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.type", "hadoop")
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", WAREHOUSE_PATH)

print(f"Catalog '{CATALOG_NAME}' registered")
print(f"Type     : Hadoop (metadata stored in Object Storage)")
print(f"Warehouse: {WAREHOUSE_PATH}")

## Step 4: Create a Database

In Iceberg, a **database** (also called a namespace) is a logical grouping of tables — similar to a schema in traditional databases.

With a Hadoop Catalog, creating a database simply creates a folder under the warehouse path:
```
oci://bucket@namespace/<WAREHOUSE_NAME>/<DATABASE_NAME>/
```

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{DATABASE_NAME}")

print(f"Database '{DATABASE_NAME}' is ready")
print(f"\nAll databases in catalog '{CATALOG_NAME}':")
spark.sql(f"SHOW DATABASES IN {CATALOG_NAME}").show()

## Step 5: Create an Iceberg Table

When creating an Iceberg table you define:
- **Schema**: column names and types
- **File format**: `USING iceberg` tells Spark this is an Iceberg table (data files will be Parquet by default)
- **Partitioning** *(optional)*: Iceberg supports hidden partitioning — you declare the partition column and Iceberg handles the physical layout automatically, without exposing partition columns to query writers

> **Note on `DROP TABLE IF EXISTS`:** The line below drops the table if it already exists so this notebook can be safely re-run. In production, remove that line to avoid accidental data loss.

In [ ]:
# Drop the table if it already exists — safe for re-running this notebook.
# Remove this line in production to avoid accidental data loss.
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")

spark.sql(f"""
    CREATE TABLE {FULL_TABLE_NAME} (
        employee_id   INT,
        employee_name STRING,
        salary        DOUBLE,
        department    STRING,
        hire_date     DATE
    )
    USING iceberg
    PARTITIONED BY (department)
""")

# Iceberg stores data as Parquet files by default.
# The 'department' partition means each department gets its own subfolder,
# so queries filtering by department skip irrelevant files entirely (partition pruning).
print(f"Table created: {FULL_TABLE_NAME}")
print(f"Location     : {WAREHOUSE_PATH}/{DATABASE_NAME}/{TABLE_NAME}")

## Step 6: Insert Sample Data

Instead of hardcoding values directly in SQL, we build a **Pandas DataFrame** first and convert it to a Spark DataFrame before writing. This pattern is more reusable — in practice you would replace the Pandas DataFrame with data loaded from a file, an API, a database, or any other source.

In [ ]:
import pandas as pd
from datetime import date

# Build your data as a Pandas DataFrame.
# Replace this with any data source: pd.read_csv(), pd.read_json(), API response, etc.
df = pd.DataFrame([
    (101, "John Doe",       75000.0, "Engineering", date(2022, 1, 15)),
    (102, "Jane Smith",     85000.0, "Sales",       date(2021, 3, 20)),
    (103, "Bob Johnson",    65000.0, "Engineering", date(2023, 6, 10)),
    (104, "Alice Williams", 95000.0, "Management",  date(2020, 8,  5)),
    (105, "Charlie Brown",  70000.0, "HR",          date(2022, 11, 30)),
    (106, "Diana Prince",   88000.0, "Engineering", date(2021, 9, 12)),
    (107, "Eve Adams",      72000.0, "Sales",       date(2023, 2, 18)),
], columns=["employee_id", "employee_name", "salary", "department", "hire_date"])

# Convert to a Spark DataFrame and write to the Iceberg table.
# Each writeTo() call is a full ACID transaction and creates a new Iceberg snapshot.
spark.createDataFrame(df).writeTo(FULL_TABLE_NAME).append()

print(f"{len(df)} rows written — this created snapshot #1")

## Step 7: Query the Table

Standard SQL works as expected. Because the table is partitioned by `department`, any `WHERE department = '...'` filter will only scan the relevant partition files — this is called **partition pruning**.

In [ ]:
# Full table scan
print("All employees:")
spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} ORDER BY department, employee_id").show()

# Filtered query — only scans the Engineering partition files
print("Engineering department (partition pruning in action):")
spark.sql(f"""
    SELECT employee_id, employee_name, salary
    FROM {FULL_TABLE_NAME}
    WHERE department = 'Engineering'
    ORDER BY salary DESC
""").show()

# Aggregation by department
print("Department summary:")
spark.sql(f"""
    SELECT department,
           COUNT(*)              AS headcount,
           ROUND(AVG(salary), 2) AS avg_salary,
           MIN(hire_date)        AS earliest_hire
    FROM {FULL_TABLE_NAME}
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

## Step 8: Schema Evolution

One of Iceberg's most important features is **safe schema evolution**. You can add, rename, or drop columns without rewriting existing data files.

Iceberg tracks schema changes in its metadata. When old files are read after a column is added, the missing column simply returns `null` (or its default value). This is safe and transparent to query writers.

In [ ]:
# Add a new column — no data rewrite needed
spark.sql(f"ALTER TABLE {FULL_TABLE_NAME} ADD COLUMN location STRING")

# Insert new data that includes the new column
spark.sql(f"""
    INSERT INTO {FULL_TABLE_NAME} VALUES
        (108, 'Frank Castle', 80000.0, 'Engineering', DATE '2024-01-10', 'New York'),
        (109, 'Grace Hopper', 91000.0, 'Management',  DATE '2019-05-22', 'Boston')
""")

# When reading, existing rows will show NULL for 'location' — no errors, no data loss
print("Table after schema evolution (location is NULL for old rows):")
spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} ORDER BY employee_id").show()

## Step 9: Snapshots and Time Travel

Every write operation in Iceberg (INSERT, UPDATE, DELETE, schema change) creates a new **snapshot**. A snapshot is an immutable, point-in-time view of the entire table.

This enables **time travel**: querying the table as it was at a specific snapshot or timestamp, without any additional setup.

In [ ]:
# List all snapshots — each write created one
snapshots_df = spark.sql(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM {FULL_TABLE_NAME}.snapshots
    ORDER BY committed_at
""")
snapshots_df.show(truncate=False)

# Get the first snapshot ID (before schema evolution and the new rows)
first_snapshot_id = snapshots_df.collect()[0]["snapshot_id"]
print(f"First snapshot ID: {first_snapshot_id}")

# Time travel: query the table as it was at the first snapshot
# At that point it had only 7 rows and no 'location' column
print("\nTable at first snapshot (7 rows, original schema):")
spark.sql(f"""
    SELECT * FROM {FULL_TABLE_NAME}
    VERSION AS OF {first_snapshot_id}
    ORDER BY employee_id
""").show()

## Step 10: Inspect Physical Files

Iceberg exposes metadata tables you can query like regular tables. The `.files` metadata table shows the actual Parquet data files that back the current snapshot.

Notice how files are organized by partition — each department gets its own directory and file(s).

In [ ]:
files_df = spark.sql(f"""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM {FULL_TABLE_NAME}.files
""")

print("Physical data files in OCI Object Storage:")
files_df.show(truncate=False)

# Summary stats
total_files   = files_df.count()
total_records = files_df.agg({"record_count": "sum"}).collect()[0][0]
total_size_kb = files_df.agg({"file_size_in_bytes": "sum"}).collect()[0][0] / 1024

print(f"Files  : {total_files}  (one per partition per write batch)")
print(f"Records: {total_records}")
print(f"Size   : {total_size_kb:.1f} KB")

## Summary

This notebook covered the end-to-end workflow for creating and using an Apache Iceberg table in OCI Object Storage with the native `oci://` protocol.

### What was demonstrated

| Step | Concept |
|---|---|
| Catalog registration | Registering a Hadoop Catalog in Spark at runtime |
| Database creation | Logical grouping of Iceberg tables |
| Table creation | Schema definition with partitioning |
| Insert | ACID transactions and snapshot creation |
| Querying | Standard SQL with partition pruning |
| Schema evolution | Adding columns without rewriting data |
| Time travel | Querying past snapshots with `VERSION AS OF` |
| Metadata inspection | Using `.files` and `.snapshots` metadata tables |

### Key takeaways

- **No S3A JARs needed**: `oci://` works natively in OCI environments
- **Authentication is automatic**: permissions are resolved automatically in AI Data Platform Workbench
- **Iceberg is a table format, not a file format**: data files are Parquet, Iceberg manages the metadata layer on top
- **Schema evolution is safe**: add or drop columns without breaking existing readers or rewriting files
- **Every write is a snapshot**: time travel comes for free

---